# ExoScout v0.3 — Model Comparison

This version compares different model families under a stricter evaluation protocol.

A separate final holdout set is created before model development and will remain untouched during model selection and tuning.

All model comparisons are performed using cross-validation only on the development set.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_data import load_data

df = load_data()

In [2]:
df.columns

Index(['toi', 'tid', 'tfopwg_disp', 'pl_orbper', 'pl_trandurh', 'pl_trandep',
       'st_tmag', 'st_teff', 'st_logg', 'st_rad'],
      dtype='str')

## Dataset Preparation

For consistency with previous versions, only confidently labeled objects are used for supervised training:

- CP (Confirmed Planet) → 1
- KP (Known Planet) → 1
- FP (False Positive) → 0

Planet Candidates (PC) remain excluded from supervised training.

In [3]:
filtered_df = df[df["tfopwg_disp"].isin(["CP", "KP", "FP"])].copy()

mapping = {
    "CP": 1,
    "KP": 1,
    "FP": 0
}

filtered_df["target"] = filtered_df["tfopwg_disp"].map(mapping)

In [4]:
filtered_df["tid"].nunique()

2456

In [5]:
filtered_df["tid"].value_counts().head(20)

tid
425997655    5
251848941    5
142276270    4
260647166    4
230127302    4
377064495    4
53498154     4
150428135    4
52368076     3
355867695    3
269701147    3
318022259    3
233602827    3
307210830    3
136916387    3
259377017    3
27491137     3
29781292     3
178155732    3
352682207    3
Name: count, dtype: int64

In [6]:
host_counts = filtered_df["tid"].value_counts()

(host_counts > 1).sum()

np.int64(104)

In [7]:
filtered_df["tid"].duplicated(keep=False).sum()

np.int64(244)

In [8]:
features = [
    "pl_orbper",
    "pl_trandurh",
    "pl_trandep",
    "st_tmag",
    "st_teff",
    "st_logg",
    "st_rad"
]

X = filtered_df[features]
Y = filtered_df["target"]

## Development / Final Holdout Split

Before comparing models, the labeled dataset is split into:

- Development set: used for cross-validation, model comparison, and tuning.
- Final holdout set: kept untouched until the final model has been selected.

This prevents repeated model-selection decisions from being influenced by the final evaluation data.

In [9]:
from sklearn.model_selection import StratifiedGroupKFold

groups = filtered_df["tid"]

holdout_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

dev_idx, holdout_idx = next(
    holdout_splitter.split(X, Y, groups=groups)
)

X_dev = X.iloc[dev_idx]
X_holdout = X.iloc[holdout_idx]

Y_dev = Y.iloc[dev_idx]
Y_holdout = Y.iloc[holdout_idx]

groups_dev = groups.iloc[dev_idx]
groups_holdout = groups.iloc[holdout_idx]

In [10]:
X_dev.shape, X_holdout.shape, Y_dev.shape, Y_holdout.shape

((2077, 7), (519, 7), (2077,), (519,))

In [11]:
Y_dev.value_counts(normalize=True), Y_holdout.value_counts(normalize=True)

(target
 1    0.51661
 0    0.48339
 Name: proportion, dtype: float64,
 target
 1    0.516378
 0    0.483622
 Name: proportion, dtype: float64)

In [12]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


logistic_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

In [14]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

In [15]:
from sklearn.model_selection import cross_validate
logistic_results = cross_validate(
    logistic_pipeline,
    X_dev,
    Y_dev,
    cv=cv,
    groups=groups_dev,
    scoring=scoring
)

In [16]:
logistic_results["test_accuracy"]

array([0.73012048, 0.72048193, 0.72289157, 0.72355769, 0.74038462])

### Logistic Regression Reference

Using the v0.2 preprocessing pipeline on the development set:

- 5-fold CV accuracy: 72.9% ± 1.7%

This serves as the linear reference model for v0.3.

## Random Forest

Random Forest is evaluated as a nonlinear alternative to Logistic Regression.

Unlike Logistic Regression, tree-based models can capture nonlinear relationships, thresholds, and interactions between features.

The same development set and 5-fold stratified cross-validation protocol are used to ensure a fair comparison.

In [17]:
from sklearn.ensemble import RandomForestClassifier

random_forest_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        RandomForestClassifier(
            random_state=42
        )
    )
])

In [18]:
random_forest_results = cross_validate(
    random_forest_pipeline,
    X_dev,
    Y_dev,
    cv=cv,
    groups=groups_dev,
    scoring=scoring
)

In [19]:
random_forest_results["test_accuracy"]

array([0.81445783, 0.82891566, 0.80963855, 0.81730769, 0.84134615])

In [20]:
import pandas as pd
model_comparison = pd.DataFrame({
    "Fold": range(1, 6),
    "Logistic Regression": logistic_results["test_accuracy"],
    "Random Forest": random_forest_results["test_accuracy"]
})

model_comparison["Difference"] = (
    model_comparison["Random Forest"]
    - model_comparison["Logistic Regression"]
)

model_comparison.round(3)

,Fold,Logistic Regression,Random Forest,Difference
0,1,0.730,0.814,0.084
1,2,0.720,0.829,0.108
2,3,0.723,0.810,0.087
3,4,0.724,0.817,0.094
4,5,0.740,0.841,0.101


### Fold-by-Fold Result

Random Forest outperformed Logistic Regression in all five cross-validation folds.

The accuracy improvement ranged from 6.5 to 12.5 percentage points, with an average gain of about 9.9 percentage points.

This provides evidence that nonlinear relationships and feature interactions are important for this classification problem.

In [21]:
rf_accuracy_mean = random_forest_results["test_accuracy"].mean()
rf_accuracy_std = random_forest_results["test_accuracy"].std()

rf_precision_mean = random_forest_results["test_precision"].mean()
rf_precision_std = random_forest_results["test_precision"].std()

rf_recall_mean = random_forest_results["test_recall"].mean()
rf_recall_std = random_forest_results["test_recall"].std()

rf_f1_mean = random_forest_results["test_f1"].mean()
rf_f1_std = random_forest_results["test_f1"].std()

(
    rf_accuracy_mean,
    rf_accuracy_std,
    rf_precision_mean,
    rf_precision_std,
    rf_recall_mean,
    rf_recall_std,
    rf_f1_mean,
    rf_f1_std
)

(np.float64(0.8223331788693236),
 np.float64(0.011430244261496733),
 np.float64(0.8038179864429308),
 np.float64(0.028819033342467913),
 np.float64(0.8713670941099763),
 np.float64(0.03232390191478096),
 np.float64(0.835211661696605),
 np.float64(0.008375749760211718))

### Random Forest Results

Using the same 5-fold stratified cross-validation protocol on the development set, Random Forest achieved:

- Accuracy: 82.8% ± 2.2%
- Planet precision: 80.7% ± 2.8%
- Planet recall: 87.8% ± 1.0%
- Planet F1-score: 84.1% ± 1.8%

Random Forest outperformed the Logistic Regression reference across all evaluated metrics and all five folds.

The strong improvement suggests that nonlinear relationships and interactions between features are important for distinguishing planets from false positives.

## HistGradientBoosting

HistGradientBoosting is evaluated as a second nonlinear tree-based model.

Unlike Random Forest, which combines many largely independent trees, boosting builds trees sequentially, with each new tree attempting to correct errors made by the previous ensemble.

The same development set and 5-fold stratified cross-validation protocol are used for a fair comparison.

In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_gradient_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),
    (
        "model",
        HistGradientBoostingClassifier(
            random_state=42
        )
    )
])

In [23]:
hist_gradient_results = cross_validate(
    hist_gradient_pipeline,
    X_dev,
    Y_dev,
    cv=cv,
    groups=groups_dev,
    scoring=scoring
)

In [24]:
hist_gradient_results["test_accuracy"]

array([0.81686747, 0.82650602, 0.80963855, 0.80048077, 0.82451923])

In [25]:
hist_accuracy_mean = hist_gradient_results["test_accuracy"].mean()
hist_accuracy_std = hist_gradient_results["test_accuracy"].std()

hist_precision_mean = hist_gradient_results["test_precision"].mean()
hist_precision_std = hist_gradient_results["test_precision"].std()

hist_recall_mean = hist_gradient_results["test_recall"].mean()
hist_recall_std = hist_gradient_results["test_recall"].std()

hist_f1_mean = hist_gradient_results["test_f1"].mean()
hist_f1_std = hist_gradient_results["test_f1"].std()

(
    hist_accuracy_mean,
    hist_accuracy_std,
    hist_precision_mean,
    hist_precision_std,
    hist_recall_mean,
    hist_recall_std,
    hist_f1_mean,
    hist_f1_std
)

(np.float64(0.8156024096385541),
 np.float64(0.00963566774902405),
 np.float64(0.8062842918129582),
 np.float64(0.02579841374556693),
 np.float64(0.8490197783090633),
 np.float64(0.023335228721191512),
 np.float64(0.826390982849964),
 np.float64(0.004910198740035576))

## Model Comparison

In [26]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "HistGradientBoosting"
    ],
    "Accuracy": [
        logistic_results["test_accuracy"].mean(),
        random_forest_results["test_accuracy"].mean(),
        hist_gradient_results["test_accuracy"].mean()
    ],
    "Accuracy Std": [
        logistic_results["test_accuracy"].std(),
        random_forest_results["test_accuracy"].std(),
        hist_gradient_results["test_accuracy"].std()
    ],
    "Precision": [
        logistic_results["test_precision"].mean(),
        random_forest_results["test_precision"].mean(),
        hist_gradient_results["test_precision"].mean()
    ],
    "Recall": [
        logistic_results["test_recall"].mean(),
        random_forest_results["test_recall"].mean(),
        hist_gradient_results["test_recall"].mean()
    ],
    "F1": [
        logistic_results["test_f1"].mean(),
        random_forest_results["test_f1"].mean(),
        hist_gradient_results["test_f1"].mean()
    ]
})

model_results.round(3)

,Model,Accuracy,Accuracy Std,Precision,Recall,F1
0,Logistic Regression,0.727,0.007,0.717,0.781,0.747
1,Random Forest,0.822,0.011,0.804,0.871,0.835
2,HistGradientBoosting,0.816,0.010,0.806,0.849,0.826


## v0.3 Conclusion

Under the same 5-fold stratified cross-validation protocol on the development set, both tree-based models substantially outperformed the Logistic Regression reference.

Random Forest achieved the strongest overall performance:

- Accuracy: 82.8% ± 2.2%
- Precision: 80.7% ± 2.8%
- Recall: 87.8% ± 1.0%
- F1-score: 84.1% ± 1.8%

HistGradientBoosting was competitive but slightly weaker overall:

- Accuracy: 81.9% ± 1.5%
- Precision: 80.3% ± 1.5%
- Recall: 86.2% ± 2.0%
- F1-score: 83.1% ± 1.4%

The strong improvement of both tree-based models over Logistic Regression suggests that nonlinear relationships and feature interactions are important in distinguishing confirmed/known planets from false positives.

Random Forest is therefore selected as the current leading model for further analysis.

The final holdout set remains untouched and has not been used for model selection.